In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [5]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

import matplotlib.pyplot as plt


import os

from ruamel.yaml import YAML
import pandas as pd
import numpy as np
import joblib

from collections import defaultdict
from itertools import chain

import click
import json

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR, MultiStepLR
from torch.utils.data import DataLoader, Dataset


from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from transformers import DataCollatorWithPadding
from transformers import RobertaTokenizer, RobertaModel

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import sys
sys.path.append('.')
from src.funcs import set_seed
from src.funcs import metric_multi
from src.funcs import get_opt_thresh, get_preds
from src.funcs import get_conf_df
from src.spec_nn_funcs import TextDFDataset, TextModelClass
from ruamel.yaml import YAML

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import RobustScaler

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier

In [7]:


conf = YAML().load(open('params.yaml'))
conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))

set_seed(conf['seed'])

In [8]:
bert_type = conf_bert['nn_bert']['bert_type']


In [9]:
VALID_BATCH_SIZE = conf_bert['nn']['batch_size']
TRAIN_BATCH_SIZE = conf_bert['nn']['batch_size']
MAX_SEQ_LENGTH = conf_bert['nn']['maxlen']

checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}


# Предсказания техник

In [10]:
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])
mlb_sub = joblib.load('data/temp/subt/ttp/mlb.pkl')

In [102]:
data_ttp = pd.read_csv(conf_ttp['feat_gen_ttp']['data_fn'])
data_ttp['target'] = data_ttp['target'].map(lambda x: eval(x))
data_ttp['ttp'] = data_ttp['ttp'].map(lambda x: eval(x))

data_sub = pd.read_csv('data/temp/subt/ttp/data_df.csv')
data_sub['target'] = data_sub['target'].map(lambda x: eval(x))
data_sub['ttp'] = data_sub['ttp'].map(lambda x: eval(x))

/tmp/ipykernel_32834/3720652998.py:5: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sub = pd.read_csv('data/temp/subt/ttp/data_df.csv')


In [15]:
model_bert_ttp = torch.load(f'{conf_bert_ttp["nn_bert_ttp"]["model_fn"]}')

model_bert_sub = torch.load(f'data/temp/subt/model.pt')

/tmp/ipykernel_32834/1293696629.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_bert_ttp = torch.load(f'{conf_bert_ttp["nn_bert_ttp"]["model_fn"]}')
/tmp/ipykernel

In [16]:
feat_ttp = pd.read_csv(conf_ttp['feat_eng_ttp']['feat_final_fn'])
feat_sub = pd.read_csv('data/temp/subt/ttp/feat_final_df.csv')



In [17]:
feat_sub.shape, feat_ttp.shape

((33698, 805), (32379, 658))

In [18]:
tr_ttp_ds = TextDFDataset(data_ttp.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_ttp_ds = TextDFDataset(data_ttp.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_ttp_ds = TextDFDataset(data_ttp.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_ttp_ld = DataLoader(tr_ttp_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_ttp_ld = DataLoader(val_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_ttp_ld = DataLoader(ts_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [19]:
tr_sub_ds = TextDFDataset(data_sub.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_sub_ds = TextDFDataset(data_sub.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_sub_ds = TextDFDataset(data_sub.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_sub_ld = DataLoader(tr_sub_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_sub_ld = DataLoader(val_sub_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_sub_ld = DataLoader(ts_sub_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [20]:
Y_ttp_val_proba = np.array(get_preds(model_bert_ttp, ld=val_ttp_ld)['pred'])
Y_ttp_tr_proba = np.array(get_preds(model_bert_ttp, ld=tr_ttp_ld)['pred'])
Y_ttp_ts_proba = np.array(get_preds(model_bert_ttp, ld=ts_ttp_ld)['pred'])

In [21]:
Y_sub_val_proba = np.array(get_preds(model_bert_sub, ld=val_sub_ld)['pred'])
Y_sub_tr_proba = np.array(get_preds(model_bert_sub, ld=tr_sub_ld)['pred'])
Y_sub_ts_proba = np.array(get_preds(model_bert_sub, ld=ts_sub_ld)['pred'])

In [22]:
thresh_ttp_l = get_opt_thresh(y_true = np.array(data_ttp.loc[data_ttp.split=='tr', 'target'].values.tolist()), 
                          probas = Y_ttp_tr_proba, mlb = mlb_ttp, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)

thresh_sub_l = get_opt_thresh(y_true = np.array(data_sub.loc[data_sub.split=='tr', 'target'].values.tolist()), 
                          probas = Y_sub_tr_proba, mlb = mlb_sub, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)



/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.11/site-packages/sklearn/m

In [23]:
ttp_df = pd.concat([data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_ttp_tr_proba.tolist()),
          data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_ttp_val_proba.tolist()),
           data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_ttp_ts_proba.tolist())
          ], axis=0, ignore_index=True)

ttp_df['pred_ttp'] = ttp_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_ttp_l)])


sub_df = pd.concat([data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_sub_tr_proba.tolist()),
          data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_sub_val_proba.tolist()),
           data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_sub_ts_proba.tolist())
          ], axis=0, ignore_index=True)

sub_df['pred_ttp'] = sub_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_sub_l)])

In [24]:
ttp_df['pred_str_ttp'] = ttp_df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])
sub_df['pred_str_ttp'] = sub_df['pred_ttp'].map(lambda x: mlb_sub.inverse_transform(np.array([x]))[0])

# Сверка

## pr_auc

In [25]:
pr_ts, pr_ts_l = metric_multi(np.array(ttp_df.query('split=="ts"')['target'].values.tolist()), 
                            np.array(ttp_df.query('split=="ts"')['proba_ttp'].values.tolist()),
                            average_precision_score)

pr_val, pr_val_l = metric_multi(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                            np.array(ttp_df.query('split=="val"')['proba_ttp'].values.tolist()),
                            average_precision_score)

pr_tr, pr_tr_l = metric_multi(np.array(ttp_df.query('split=="tr"')['target'].values.tolist()), 
                            np.array(ttp_df.query('split=="tr"')['proba_ttp'].values.tolist()),
                            average_precision_score)


pr_tr, pr_val, pr_ts

(0.8269416975442622, 0.5719118389835383, 0.562590359932677)

In [26]:
pr_ts, pr_ts_l = metric_multi(np.array(sub_df.query('split=="ts"')['target'].values.tolist()), 
                            np.array(sub_df.query('split=="ts"')['proba_ttp'].values.tolist()),
                            average_precision_score)

pr_val, pr_val_l = metric_multi(np.array(sub_df.query('split=="val"')['target'].values.tolist()), 
                            np.array(sub_df.query('split=="val"')['proba_ttp'].values.tolist()),
                            average_precision_score)


pr_tr, pr_tr_l = metric_multi(np.array(sub_df.query('split=="tr"')['target'].values.tolist()), 
                            np.array(sub_df.query('split=="tr"')['proba_ttp'].values.tolist()),
                            average_precision_score)



pr_tr, pr_val, pr_ts

(0.817115467073494, 0.501528070274588, 0.5212990674878488)

## f1

In [33]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split=="val"')['pred_ttp'].values.tolist()), average='micro')

p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split=="val"')['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(0.6067165366862963, 0.5265788800947466)

In [34]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(sub_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(sub_df.query('split=="val"')['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(sub_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(sub_df.query('split=="val"')['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(0.5522865366165653, 0.4274561424537244)

### для субтехник не считаем ошибкой все, что до точки

In [35]:
sub_df = pd.concat([data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_sub_tr_proba.tolist()),
          data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_sub_val_proba.tolist()),
           data_sub[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_sub_ts_proba.tolist())
          ], axis=0, ignore_index=True)

sub_df['pred_ttp'] = sub_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_sub_l)])
sub_df['pred_str_ttp'] = sub_df['pred_ttp'].map(lambda x: mlb_sub.inverse_transform(np.array([x]))[0])

In [36]:
Y_val_proba = Y_sub_val_proba
thresh_l = thresh_sub_l
Y_val = np.array(data_sub.loc[data_sub.split=='val', 'target'].values.tolist())

In [37]:
res_df = pd.DataFrame()
res_df['y'] = Y_val.tolist()
res_df['y_proba'] = Y_val_proba.tolist()

thresh_col = 'y_p'

res_df[thresh_col] = res_df['y_proba'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_l)])


error_df = data_sub.query('split=="val"').reset_index(names='val_idx').join(res_df[['y', 'y_proba', thresh_col]])
error_df['prob_label'] = mlb_sub.inverse_transform(np.array(error_df[thresh_col].tolist()))

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(res_df['y'].values.tolist()), 
                                                    np.array(res_df[thresh_col].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(res_df['y'].values.tolist()), 
                                                    np.array(res_df[thresh_col].values.tolist()), average='macro')


f1_val_micro, f1_val_macro

/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(0.5522865366165653, 0.4274561424537244)

In [38]:
t_l = [[it.split('.')[0]] for it in data_sub.explode('ttp')['ttp'].dropna().unique()]

In [43]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_f1 = MultiLabelBinarizer()
mlb_f1.fit(t_l)


MultiLabelBinarizer()

In [44]:
error_df['pred_enc'] = error_df['prob_label'].map(lambda x: [it.split('.')[0] for it in x]).tolist()
error_df['pred_enc'] = mlb_f1.transform(error_df['pred_enc']).tolist()

error_df['ttp_enc'] = error_df['ttp'].map(lambda x: [it.split('.')[0] for it in x]).tolist()
error_df['ttp_enc'] = mlb_f1.transform(error_df['ttp_enc']).tolist()

In [45]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(error_df['ttp_enc'].values.tolist()), 
                                                    np.array(error_df['pred_enc'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(error_df['ttp_enc'].values.tolist()), 
                                                    np.array(error_df['pred_enc'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(0.5866231647634584, 0.4889006130861287)

In [42]:
_, res_l = metric_multi(np.array(error_df['ttp_enc'].tolist()), np.array(error_df['pred_enc'].tolist()), f1_score)

pd.DataFrame({'qual':res_l, 'class':mlb_f1.classes_}).sort_values(by='qual').head(20)


,qual,class
15,0.000000,T1030
96,0.000000,T1557
58,0.000000,T1133
22,0.000000,T1048
112,0.000000,T1584
7,0.000000,T1014
33,0.000000,T1072
121,0.000000,T1622
117,0.000000,T1598
118,0.000000,T1608


# Анализ

In [32]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()

ttp,T1001,T1003,T1005,T1007,T1008,T1010,T1012,T1014,T1016,T1018,...,T1591,T1592,T1593,T1595,T1598,T1608,T1614,T1620,T1622,rare
split,,,,,,,,,,,,,,,,,,,,,
tr,41,190,168,41,37,22,89,81,225,58,...,55,65,55,84,28,52,33,48,66,345
ts,10,47,42,10,9,5,22,5,57,15,...,4,2,4,6,6,13,8,4,3,93
val,9,49,42,10,9,5,22,4,55,15,...,3,3,3,4,7,12,8,4,3,87


## Выборки одинаковые

In [33]:
data_ttp.groupby('split').size()

split
tr     22142
ts      5118
val     5119
dtype: int64

In [34]:
data_sub.groupby('split').size()

split
tr     23461
ts      5118
val     5119
dtype: int64

In [35]:
ttp_df.loc[(ttp_df.split!="tr") , ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split!="tr"), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='inner'
)

,sentence,ttp_x,labels_x,pred_str_ttp_x,ttp_y,labels_y,pred_str_ttp_y
0,Running code in the context of another process...,[T1055],"['defense-evasion', 'privilege-escalation']","(T1055,)",[rare],"['defense-evasion', 'privilege-escalation']","(rare,)"
1,"Adversaries may also create ""hidden"" scheduled...",[T1053],"['execution', 'persistence', 'privilege-escala...","(T1070, T1564)",[T1053.005],"['execution', 'persistence', 'privilege-escala...","(rare,)"
2,Adversaries may attach filters to a network so...,[T1205],"['defense-evasion', 'persistence', 'command-an...","(T1205, T1562)",[rare],"['defense-evasion', 'persistence', 'command-an...","(T1205, rare)"
3,"Adversaries may store data in ""fileless"" forma...",[T1027],['defense-evasion'],"(T1564,)",[T1027.011],['defense-evasion'],"(T1564.004,)"
4,Some forms of fileless storage activity may in...,[T1027],['defense-evasion'],(),[T1027.011],['defense-evasion'],()
...,...,...,...,...,...,...,...
10232,The attackers use the Rclone synchronization u...,[T1567],['exfiltration'],"(T1567,)",[T1567.002],['exfiltration'],"(T1567.002,)"
10233,"BlackCat stops security, backup, database, ema...",[T1489],['impact'],"(T1489,)",[T1489],['impact'],"(T1489,)"
10234,Access Token Manipulation: Token Impersonation...,[T1134],"['defense-evasion', 'privilege-escalation']",(),[T1134.001],"['defense-evasion', 'privilege-escalation']",()
10235,SEABORGIUM and TA453 actors use online data se...,[T1589],['reconnaissance'],(),[rare],['reconnaissance'],()


# Bert by class сравнить

In [46]:
by_ttp_df = pd.read_csv('data/out/bert_ttp/bert_by_class_metric.csv')
by_sub_df = pd.read_csv('data/temp/subt/bert_by_class_metric.csv')


In [47]:
diff_classes = by_sub_df.merge(by_ttp_df, on='class').assign(diff=lambda x: abs(x['qual_x']-x['qual_y'])).sort_values(by='diff', ascending=False)['class'].head(10)
by_sub_df.merge(by_ttp_df, on='class').assign(diff=lambda x: abs(x['qual_x']-x['qual_y'])).sort_values(by='diff', ascending=False).head(10)

,qual_x,class,qual_y,diff
3,0.000000,T1529,0.857143,0.857143
6,0.000000,T1614,0.800000,0.800000
8,0.000000,T1134,0.733333,0.733333
0,0.000000,T1059,0.701754,0.701754
1,0.000000,T1071,0.690141,0.690141
70,0.666667,T1221,0.000000,0.666667
9,0.000000,T1003,0.632653,0.632653
13,0.181818,T1036,0.741259,0.559441
42,0.500000,T1040,0.000000,0.500000
17,0.222222,T1554,0.666667,0.444444


# main_diff

In [78]:
N = 1
ttp = diff_classes.iloc[N]
ttp

'T1614'

In [79]:
by_sub_df[by_sub_df['class'].map(lambda x: ttp in x)]

,qual,class
21,0.000000,T1614
129,0.545455,T1614.001


In [80]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     33
ts      8
val     8
Name: T1614, dtype: int64

In [81]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     48
ts      3
val     3
Name: T1614, dtype: int64

In [82]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split=="val") & (sub_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='outer'
)

,sentence,ttp_x,labels_x,pred_str_ttp_x,ttp_y,labels_y,pred_str_ttp_y
0,"During [Operation Dream Job], [Lazarus Group] ...",[T1614],['discovery'],(),NaN,NaN,NaN
1,XCSSET uses AppleScript to check the host's la...,[T1614],['discovery'],"(T1082, T1614)",NaN,NaN,NaN
2,Avaddon checks for specific keyboard layouts a...,[T1614],['discovery'],"(T1614,)",NaN,NaN,NaN
3,Bazar can perform a check to ensure that the o...,[T1614],['discovery'],"(T1614,)",NaN,NaN,NaN
4,Clop has checked the keyboard language using t...,[T1614],['discovery'],"(T1614,)",NaN,NaN,NaN
5,"Before executing malicious code, [Ragnar Locke...",[T1614],['discovery'],"(T1480, T1497, T1614)",[T1614],['discovery'],"(T1480, T1614.001)"
6,Crimson can identify the geographical location...,[T1614],['discovery'],"(T1016, T1614)",[T1614],['discovery'],()
7,Raccoon Stealer v2 collects the time zone info...,[T1614],['discovery'],"(T1124,)",[T1614],['discovery'],"(T1124,)"


In [53]:
data = pd.read_csv(conf['prep_text']['prep_fn'])


data['ttp'] = data['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20


sel_dop = (data.split=='tr')
mini_ttp_l = data[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

True

In [54]:
data_s= pd.read_csv('data/temp/subt/ttp/prep_df.csv')

data_s['ttp'] = data_s['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20

sel_dop = (data_s.split=='tr')
mini_ttp_l = data_s[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

True

## найдем thresh

In [128]:
ttp_idx = np.where(mlb_ttp.classes_==ttp)[0][0]
sub_idx = np.where(mlb_sub.classes_==ttp)[0][0]

In [129]:
thresh_ttp_l[ttp_idx], thresh_sub_l[sub_idx]

(0.165, 0.405)

<div class='alert alert-info'> 
    
    - T1071 - с подтехниками, сам класс плохо определяется на уровне субтехник, но его подтехники нормально
    - T1614 - с подтехниками чистый класс аугментировался, и хуже предсказался, хотя в одном случае - T1614.001 (но формально это ошибка). Хотя все подклассы объединены в метку T1614, поэтому даже, где техника кажется правильной, может быть подтехника (это те, где nan в пересечении ttp_df и sub_df)
    - T1134 как и T1614, хотя f1  ноль для чистого, в 2 случаях субкласс предсказался, хотя формально и ошибка, алгоритм почуял верно. T1134 интересно, что теряем подтехники маленькие (в rare), хотя могли их отнести к классу
    - T1036 аналогично
    - T1059 - если считать не только чистый, но и подтехники, то качество норма
    - T1071 - f1 за счет подтехник, которые да, лучше, чем на уровне подтехник предсказываются
    - T1221, T1040 - аугментация в пользу субтехник
    - T1554 - на один пример лучше предсказано для техник, аугментация поточнее или другие факторы
</div>

<div class='alert alert-info'> TO DO
    
    - T1134 - теряем подтехники маленькие (в rare), хотя могли их отнести к классу. Может так  сделать?
    - может вывести для субтехник метрику f1 для классов в предположении, что target до точки и предсказание, чтобы не считать ошибки на уровне субтехник
</div>

# T1529

In [93]:
ttp = 'T1529'

In [94]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     56
ts      3
val     4
Name: T1529, dtype: int64

In [95]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     48
ts      3
val     4
Name: T1529, dtype: int64

In [96]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split=="val") & (sub_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='outer'
)

,sentence,ttp_x,labels_x,pred_str_ttp_x,ttp_y,labels_y,pred_str_ttp_y
0,Shutting down or rebooting systems may disrupt...,[T1529],['impact'],"(T1529,)",[T1529],['impact'],()
1,Maze has issued a shutdown command on a victim...,[T1529],['impact'],"(T1529,)",[T1529],['impact'],()
2,KillDisk attempts to reboot the machine by ter...,[T1529],['impact'],"(T1489, T1562)",[T1529],['impact'],"(T1489,)"
3,Fantasy reboots the system after completing it...,[T1529],['impact'],"(T1070, T1529)",[T1529],['impact'],"(T1070.004,)"


In [98]:
data = pd.read_csv(conf['prep_text']['prep_fn'])


data['ttp'] = data['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20


sel_dop = (data.split=='tr')
mini_ttp_l = data[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

True

In [100]:
data_s= pd.read_csv('data/temp/subt/ttp/prep_df.csv')

data_s['ttp'] = data_s['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20

sel_dop = (data_s.split=='tr')
mini_ttp_l = data_s[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

True

In [103]:
data_s[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num][ttp]

13

То есть 13 кейсов только

## найдем thresh

In [104]:
ttp_idx = np.where(mlb_ttp.classes_==ttp)[0][0]
sub_idx = np.where(mlb_sub.classes_==ttp)[0][0]

In [105]:
thresh_ttp_l[ttp_idx], thresh_sub_l[sub_idx]

(0.135, 0.495)

<div class='alert alert-info'>
и train выборки и границы сильно отличаются
</div>

In [107]:
conf_bert_ttp['nn_bert_ttp']['opt_metric_fn']

'data/out/bert_ttp/bert_opt_metric.csv'

In [116]:
opt_metric_sub_df = pd.read_csv('data/temp/subt/ttp/bert_opt_metric.csv').set_index('Unnamed: 0').loc[lambda x:x['class_nm']==ttp]
idx_sub = opt_metric_sub_df['f1'].argmax()

In [ ]:
opt_metric_df = pd.read_csv(conf_bert_ttp['nn_bert_ttp']['opt_metric_fn']).set_index('Unnamed: 0').loc[lambda x:x['class_nm']==ttp]

idx = opt_metric_df['f1'].argmax()

In [115]:
opt_metric_df.iloc[idx-2:idx+2]


,precision,recall,f1,sup,class_nm
Unnamed: 0,,,,,
0.131,0.931034,0.964286,0.947368,56,T1529
0.133,0.931034,0.964286,0.947368,56,T1529
0.135,0.947368,0.964286,0.955752,56,T1529
0.137,0.947368,0.964286,0.955752,56,T1529


In [120]:
opt_metric_sub_df.iloc[60:70]

,precision,recall,f1,sup,class_nm
Unnamed: 0,,,,,
0.121,0.718750,0.958333,0.821429,48,T1529
0.123,0.718750,0.958333,0.821429,48,T1529
0.125,0.718750,0.958333,0.821429,48,T1529
0.127,0.714286,0.937500,0.810811,48,T1529
0.129,0.714286,0.937500,0.810811,48,T1529
0.131,0.714286,0.937500,0.810811,48,T1529
0.133,0.714286,0.937500,0.810811,48,T1529
0.135,0.714286,0.937500,0.810811,48,T1529
0.137,0.714286,0.937500,0.810811,48,T1529


In [119]:
# FP, видимо, были до 
opt_metric_sub_df.iloc[idx_sub-2:idx_sub+2]


,precision,recall,f1,sup,class_nm
Unnamed: 0,,,,,
0.491,0.956522,0.916667,0.936170,48,T1529
0.493,0.956522,0.916667,0.936170,48,T1529
0.495,0.977778,0.916667,0.946237,48,T1529
0.497,0.977778,0.916667,0.946237,48,T1529


In [159]:
comp_df = ttp_df.loc[(ttp_df.ttp.map(lambda x: ttp in x))&(ttp_df.split=="tr"), ['sentence','proba_ttp', 'pred_str_ttp']]\
        .assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[ttp_idx]))\
    .merge(
sub_df.loc[(sub_df.ttp.map(lambda x: ttp in x))&(sub_df.split=="tr"), ['sentence','proba_ttp', 'pred_str_ttp']]\
        .assign(proba_ttp=lambda x:x['proba_ttp'].map(lambda y:y[sub_idx])), on='sentence', how='outer'
        
    )

In [161]:
comp_df.dropna()

,sentence,proba_ttp_x,pred_str_ttp_x,proba_ttp_y,pred_str_ttp_y
8,Adversaries may shutdown/reboot systems to int...,0.869917,"(T1529,)",0.969553,"(T1529, rare)"
17,Adversaries may attempt to shutdown/reboot a s...,0.511756,"(T1529,)",0.827930,"(T1529,)"
22,AcidRain reboots the target system once the va...,0.477181,"(T1070, T1529)",0.328643,()
25,[Olympic Destroyer] will shut down the comprom...,0.821122,"(T1070, T1529)",0.709522,"(T1529,)"
34,APT38 has used a custom MBR wiper named BOOTWR...,0.532743,"(T1529,)",0.648198,"(T1529,)"
39,APT37 has used malware that will issue the com...,0.906434,"(T1529,)",0.916333,"(T1529,)"
40,LookBack can shutdown and reboot the victim ma...,0.151101,"(T1529,)",0.111444,()
42,NotPetya will reboot the system one hour after...,0.750593,"(T1529,)",0.507124,"(T1529,)"
51,[Lazarus Group] has rebooted systems after des...,0.603689,"(T1529,)",0.975447,"(T1529,)"
52,Shamoon will reboot the infected system once t...,0.753536,"(T1070, T1529)",0.897982,"(T1529,)"


# T1124

In [61]:
ttp = 'T1124'

In [62]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     55
ts     14
val    13
Name: T1124, dtype: int64

In [63]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     55
ts     14
val    13
Name: T1124, dtype: int64

In [64]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split=="val") & (sub_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='outer'
)

,sentence,ttp_x,labels_x,pred_str_ttp_x,ttp_y,labels_y,pred_str_ttp_y
0,System time information may be gathered in a n...,[T1124],['discovery'],"(T1082, T1124)",[T1124],['discovery'],"(T1124,)"
1,"In addition, system calls – such as <code>time...",[T1124],['discovery'],"(T1124,)",[T1124],['discovery'],"(T1124,)"
2,SombRAT can execute <code>getinfo</code> to di...,[T1124],['discovery'],"(T1124,)",[T1124],['discovery'],"(T1124,)"
3,BADHATCH can obtain the `DATETIME` and `UPTIME...,[T1124],['discovery'],"(T1124,)",[T1124],['discovery'],()
4,Higaisa used a function to gather the current ...,[T1124],['discovery'],"(T1124,)",[T1124],['discovery'],"(T1124,)"
5,Crimson has the ability to determine the date ...,[T1124],['discovery'],"(T1124,)",[T1124],['discovery'],"(T1124,)"
6,TajMahal has the ability to determine local ti...,[T1124],['discovery'],"(T1124,)",[T1124],['discovery'],"(T1124,)"
7,PowerDuke has commands to get the time the mac...,[T1124],['discovery'],"(T1082, T1124)",[T1124],['discovery'],"(T1082, T1124)"
8,FELIXROOT gathers the time zone information fr...,[T1124],['discovery'],"(T1124,)",[T1124],['discovery'],"(T1124,)"
9,BendyBear has the ability to determine local t...,[T1124],['discovery'],"(T1124,)",[T1124],['discovery'],"(T1124,)"


# T1010

In [89]:
ttp = 'T1010'

In [90]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     22
ts      5
val     5
Name: T1010, dtype: int64

In [91]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     22
ts      5
val     5
Name: T1010, dtype: int64

In [92]:
by_sub_df.merge(by_ttp_df, on='class').assign(diff=lambda x: abs(x['qual_x']-x['qual_y'])).sort_values(by='diff', ascending=False).loc[lambda x: x['class']=='T1010']

,qual_x,class,qual_y,diff
77,0.727273,T1010,0.666667,0.060606


<div class='alert alert-info'> Качество сопоставимое
</div>

# T1014

In [83]:
ttp = 'T1014'

In [84]:
data_ttp.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     81
ts      5
val     4
Name: T1014, dtype: int64

In [85]:
data_sub.explode('ttp').groupby(['split', 'ttp']).size().unstack()[ttp]

split
tr     56
ts      5
val     4
Name: T1014, dtype: int64

In [86]:
ttp_df.loc[(ttp_df.split=="val") & (ttp_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']].merge(
    sub_df.loc[(sub_df.split=="val") & (sub_df.ttp.map(lambda x: ttp in x)), ['sentence', 'ttp', 'labels', 'pred_str_ttp']], on='sentence', how='outer'
)

,sentence,ttp_x,labels_x,pred_str_ttp_x,ttp_y,labels_y,pred_str_ttp_y
0,Rocke has modified /etc/regexp_domain to hook ...,[T1014],['defense-evasion'],(),[T1014],['defense-evasion'],"(rare,)"
1,Hikit is a Rootkit that has been used by Axiom.,[T1014],['defense-evasion'],(),[T1014],['defense-evasion'],"(T1587.001,)"
2,[Caterpillar WebShell] has a module to use a r...,[T1014],['defense-evasion'],(),[T1014],['defense-evasion'],()
3,The user-to-kernel module of Lazarus can turn ...,[T1014],['defense-evasion'],"(T1562,)",[T1014],['defense-evasion'],()


- тут разные train выборки для этого класса, так как без субтехник класс попал в 20-ку тех, где аугментация применяется

In [ ]:
data = pd.read_csv(conf['prep_text']['prep_fn'])


data['ttp'] = data['ttp'].map(lambda x: eval(x))
synth_thresh_class_num = 20


sel_dop = (data.split=='tr')
mini_ttp_l = data[sel_dop].explode('ttp').groupby('ttp').size().loc[lambda x: x<=synth_thresh_class_num].index.tolist()

ttp in mini_ttp_l

# Причины

Причины расхождений:
-  Основная. Разные выборки - ВСЕ, так как шла стратификация (выравнивание выборок) по разным целям (для субтехник выравнивание по субтехникам, а после их обобщения до техник, шло выравнивание выборок по техникам). В какую-то попали более удачные записи, и пара записей сильно меняла процентное соотношение, а качество предсказаний для всех записей примерно одинаковое, что подтверждается при унификации выборок. Обновленные файлы высылаю.
- Для чистых классов, где-то 0 в f1, хотя подтехники не плохо определяются (например, T1071, T1059). Чтобы повысить справедливость сравнения, убрал ошибку, связанную с угадыванием подтехники, но неугадыванием самой техники (так как формально для субтехник это ошибка, а для модели по техникам - нет).
- 20-ка самых малых классов для аугментации тоже по-разному определена, где-то из-за этого разный cut-off на train, соответственно, разные результаты на валидации.
- Общие слои в берте по-разному обучаются, так как таргет разный, выборки и пакеты 

После унификации выборок:
- f1_val_micro, f1_val_macro для техник - (0.6067165366862963, 0.5265788800947466)
- f1_val_micro, f1_val_macro для субтехник - (0.5522865366165653, 0.4274561424537244). После того, как убрал ошибку при неугадывании техники, но угадывании подтехники - (0.5866231647634584, 0.4889006130861287)

In [87]:
_, res_l = metric_multi(np.array(error_df['ttp_enc'].tolist()), np.array(error_df['pred_enc'].tolist()), f1_score)

pd.DataFrame({'qual':res_l, 'class':mlb_f1.classes_}).sort_values(by='qual').loc[lambda x: x['class'].isin(["T1059", "T1071", "T1014"])]


,qual,class
7,0.000000,T1014
28,0.603175,T1059
32,0.682540,T1071


In [88]:
pd.DataFrame({'qual':res_l, 'class':mlb_f1.classes_}).sort_values(by='qual').loc[lambda x: x['class'].isin(["T1040", "T1072", "T1014", "T1037", "T1221"])]

,qual,class
7,0.000000,T1014
33,0.000000,T1072
18,0.500000,T1040
73,0.666667,T1221
